# 05 — LLM Insights
**AutoAnalyst | Finance Module**

Uses Llama 3 (local via Ollama) to interpret EDA results and generate:
1. Executive summary
2. Key insights with specific numbers
3. Anomalies and red flags
4. Recommended next steps
5. Interactive chat — ask follow-up questions about your data

**Important:** The LLM never sees raw data.
It only receives the structured `eda_summary` dictionary from notebook 03.
This keeps prompts small, fast, and focused.

In [29]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import json
import warnings
import ollama
warnings.filterwarnings('ignore')

# Verify Ollama is running and model exists
try:
    models = [m.model for m in ollama.list().models]
    print(f'✅ Ollama connected.')
    print(f'   Available models: {models}')
except Exception as e:
    print(f'❌ Ollama not running. Start it with: ollama serve')
    print(f'   Error: {e}')

✅ Ollama connected.
   Available models: ['llama3:latest']


In [30]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
DATASET_FILENAME = 'Annual_P_L_1_final.csv'
BASE_PATH        = r'R:\AutoAnalyst\finance_module\datasets'
DATASET_PATH     = os.path.join(BASE_PATH, DATASET_FILENAME)
MODEL            = 'llama3:latest'

print(f'Dataset : {DATASET_FILENAME}')
print(f'Model   : {MODEL}')
print(f'Exists  : {os.path.exists(DATASET_PATH)}')

Dataset : Annual_P_L_1_final.csv
Model   : llama3:latest
Exists  : True


In [31]:
# ── Cell 3: Full Pipeline (self-contained) ────────────────────────────────────

def smart_load(filepath):
    raw = pd.read_csv(filepath, nrows=3, header=0)
    first_val = str(raw.iloc[0, 0]).strip().lower()
    if first_val in ['ticker', 'date', 'symbol', 'name', 'description']:
        df = pd.read_csv(filepath, skiprows=[1, 2], header=0)
        df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
    else:
        df = pd.read_csv(filepath, header=0)
    return df

def detect_finance_type(df):
    cols_str = ' '.join([c.lower().strip() for c in df.columns])
    scores = {
        'timeseries'   : sum(1 for kw in ['close','open','high','low','volume','price'] if kw in cols_str),
        'transactional': sum(1 for kw in ['amount','card','exp type','city','gender'] if kw in cols_str),
        'fundamental'  : sum(1 for kw in ['sales','profit','eps','bse','nse','market cap','ratio','ebitda'] if kw in cols_str)
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'

def universal_clean(df):
    df = df.copy()
    df.drop_duplicates(inplace=True)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace(r'[^\w]', '_', regex=True))
    return df

def clean_timeseries(df):
    df = df.copy()
    date_col = 'date' if 'date' in df.columns else df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df.dropna(subset=[date_col], inplace=True)
    df.sort_values(date_col, inplace=True)
    df.reset_index(drop=True, inplace=True)
    num_cols = [c for c in df.columns if c != date_col]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df[num_cols] = df[num_cols].ffill()
    df['daily_return']   = df['close'].pct_change() * 100
    df['ma_7']           = df['close'].rolling(7).mean()
    df['ma_30']          = df['close'].rolling(30).mean()
    df['ma_90']          = df['close'].rolling(90).mean()
    df['volatility_30d'] = df['daily_return'].rolling(30).std() * np.sqrt(252)
    df['year']           = df['date'].dt.year
    return df

def clean_transactional(df):
    df = df.copy()
    date_col = next((c for c in df.columns if 'date' in c.lower()), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
        df.sort_values(date_col, inplace=True)
        df.reset_index(drop=True, inplace=True)
        df['month']       = df[date_col].dt.month
        df['year']        = df[date_col].dt.year
        df['day_of_week'] = df[date_col].dt.day_name()
    city_col = next((c for c in df.columns if 'city' in c.lower()), None)
    if city_col:
        df[city_col] = df[city_col].str.replace(', India', '', regex=False).str.strip()
    for col in ['exp_type', 'card_type', 'gender']:
        if col in df.columns:
            df[col] = df[col].str.title()
    if 'index' in df.columns:
        df.drop(columns=['index'], inplace=True)
    return df

def clean_fundamental(df):
    df = df.copy()
    if 'join_key' in df.columns:
        df.drop(columns=['join_key'], inplace=True)
    cols_to_drop = df.isnull().mean()[df.isnull().mean() > 0.6].index.tolist()
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
    non_numeric = ['name', 'nse_code', 'industry']
    for col in [c for c in df.columns if c not in non_numeric and df[c].dtype == object]:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().sum() > df[col].notna().sum() * 0.5:
            df[col] = converted
    return df

def build_eda_summary(df, dataset_type, filename):
    """Rebuilds the eda_summary from df_clean — same as notebook 03 output."""
    desc_stats = df.select_dtypes(include=[np.number]).describe().to_dict()
    num_df     = df.select_dtypes(include=[np.number])
    flag_cols  = [c for c in num_df.columns if c.startswith('is_') or c.endswith('_missing')]
    num_df     = num_df.drop(columns=flag_cols, errors='ignore')
    corr       = num_df.corr()
    corr_pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                  .stack().reset_index())
    corr_pairs.columns = ['col_a', 'col_b', 'correlation']
    corr_pairs['abs_corr'] = corr_pairs['correlation'].abs()
    top_corr = corr_pairs.sort_values('abs_corr', ascending=False).head(5)

    domain = {}
    if dataset_type == 'timeseries':
        returns  = df['daily_return'].dropna()
        latest   = df.iloc[-1]
        trend    = 'BULLISH' if latest['close'] > latest['ma_30'] else 'BEARISH'
        yearly   = df.groupby('year').apply(
            lambda x: round((x['close'].iloc[-1] - x['close'].iloc[0]) / x['close'].iloc[0] * 100, 2)
        ).to_dict()
        domain = {
            'date_range'           : f"{df['date'].min().date()} to {df['date'].max().date()}",
            'total_trading_days'   : len(df),
            'current_price'        : round(latest['close'], 2),
            'trend_signal'         : trend,
            'avg_daily_return_pct' : round(returns.mean(), 4),
            'volatility_pct'       : round(returns.std(), 4),
            'best_day'             : str(df.loc[returns.idxmax(), 'date'].date()),
            'best_day_return_pct'  : round(returns.max(), 2),
            'worst_day'            : str(df.loc[returns.idxmin(), 'date'].date()),
            'worst_day_return_pct' : round(returns.min(), 2),
            'positive_days'        : int((returns > 0).sum()),
            'negative_days'        : int((returns < 0).sum()),
            'all_time_high'        : round(df['high'].max(), 2),
            'all_time_high_date'   : str(df.loc[df['high'].idxmax(), 'date'].date()),
            'all_time_low'         : round(df['low'].min(), 2),
            'all_time_low_date'    : str(df.loc[df['low'].idxmin(), 'date'].date()),
            'total_return_pct'     : round((latest['close'] - df['close'].iloc[0]) / df['close'].iloc[0] * 100, 2),
            'avg_daily_volume'     : int(df['volume'].mean()),
            'current_ma_7'         : round(float(df['ma_7'].iloc[-1]), 2),
            'current_ma_30'        : round(float(df['ma_30'].iloc[-1]), 2),
            'current_ma_90'        : round(float(df['ma_90'].iloc[-1]), 2),
            'price_vs_ma30'        : 'above' if df['close'].iloc[-1] > df['ma_30'].iloc[-1] else 'below',
            'annual_returns'       : yearly
        }
    elif dataset_type == 'transactional':
        total   = df['amount'].sum()
        cat_pct = (df.groupby('exp_type')['amount'].sum() / total * 100).round(2).to_dict()
        card_pct= (df.groupby('card_type')['amount'].sum() / total * 100).round(2).to_dict()
        gender_pct=(df.groupby('gender')['amount'].sum() / total * 100).round(2).to_dict()
        top_city= df.groupby('city')['amount'].sum().idxmax()
        domain = {
            'total_transactions' : len(df),
            'total_spend'        : round(total, 2),
            'avg_transaction'    : round(df['amount'].mean(), 2),
            'median_transaction' : round(df['amount'].median(), 2),
            'date_range'         : f"{df['date'].min().date()} to {df['date'].max().date()}",
            'top_category'       : max(cat_pct, key=cat_pct.get),
            'top_category_pct'   : max(cat_pct.values()),
            'top_city'           : top_city,
            'category_breakdown' : cat_pct,
            'card_breakdown'     : card_pct,
            'gender_breakdown'   : gender_pct,
        }
    elif dataset_type == 'fundamental':
        profit_col = next((c for c in ['net_profit','profit_after_tax'] if c in df.columns), None)
        profitable = int((df[profit_col] > 0).sum()) if profit_col else 'N/A'
        loss_making= int((df[profit_col] < 0).sum()) if profit_col else 'N/A'
        top5_ind   = df['industry'].value_counts().head(5).to_dict() if 'industry' in df.columns else {}
        domain = {
            'n_companies'    : len(df),
            'n_industries'   : df['industry'].nunique() if 'industry' in df.columns else 'N/A',
            'profitable_pct' : round(profitable / len(df) * 100, 1) if profit_col else 'N/A',
            'loss_making'    : loss_making,
            'top_industries' : top5_ind,
            'largest_company': df.loc[df['market_capitalization'].idxmax(), 'name'] if 'market_capitalization' in df.columns else 'N/A',
            'total_market_cap': round(df['market_capitalization'].sum(), 0) if 'market_capitalization' in df.columns else 'N/A',
        }

    return {
        'filename'        : filename,
        'dataset_type'    : dataset_type,
        'shape'           : df.shape,
        'top_correlations': top_corr[['col_a','col_b','correlation']].values.tolist(),
        'descriptive_stats': {
            col: {'mean': round(v.get('mean',0),4), 'std': round(v.get('std',0),4),
                  'min': round(v.get('min',0),4),   'max': round(v.get('max',0),4),
                  'median': round(v.get('50%',0),4)}
            for col, v in desc_stats.items()
        },
        'domain_analysis' : domain
    }

# Run pipeline
df_raw       = smart_load(DATASET_PATH)
dataset_type = detect_finance_type(df_raw)
df_clean     = universal_clean(df_raw)
if dataset_type == 'timeseries':
    df_clean = clean_timeseries(df_clean)
elif dataset_type == 'transactional':
    df_clean = clean_transactional(df_clean)
elif dataset_type == 'fundamental':
    df_clean = clean_fundamental(df_clean)

eda_summary = build_eda_summary(df_clean, dataset_type, DATASET_FILENAME)

print(f'✅ Pipeline complete | Type: {dataset_type.upper()} | Shape: {df_clean.shape}')
print(f'   EDA summary keys: {list(eda_summary.keys())}')

✅ Pipeline complete | Type: FUNDAMENTAL | Shape: (4668, 57)
   EDA summary keys: ['filename', 'dataset_type', 'shape', 'top_correlations', 'descriptive_stats', 'domain_analysis']


In [32]:
# ── Cell 4: Prompt Builder ────────────────────────────────────────────────────
# Builds a tight, structured prompt from the eda_summary.
# We never dump raw data — only the computed summary.
# The system prompt gives the LLM a role and strict output format.

SYSTEM_PROMPT = """You are a senior financial data analyst with 15 years of experience.
You receive structured summaries of financial datasets and produce sharp, specific insights.

Rules you always follow:
- Always reference specific numbers from the data provided
- Never make generic statements like 'the data shows interesting patterns'
- Be direct and specific — name exact values, dates, percentages
- Flag anything unusual or worth investigating
- Keep language professional but accessible
- Format your response with clear section headers"""


def build_prompt(eda_summary):
    dtype  = eda_summary['dataset_type']
    domain = eda_summary['domain_analysis']
    stats  = eda_summary['descriptive_stats']
    corr   = eda_summary['top_correlations']
    shape  = eda_summary['shape']

    # Format correlation pairs cleanly
    corr_str = '\n'.join(
        [f'  - {r[0]} vs {r[1]}: {round(r[2], 3)}' for r in corr[:5]]
    )

    # Format domain analysis cleanly
    domain_str = json.dumps(domain, indent=2, default=str)

    # Format key stats (only most important columns)
    key_stat_cols = list(stats.keys())[:6]
    stats_str = '\n'.join([
        f'  - {col}: mean={v["mean"]}, std={v["std"]}, min={v["min"]}, max={v["max"]}'
        for col, v in stats.items() if col in key_stat_cols
    ])

    prompt = f"""I have analyzed a {dtype.upper()} financial dataset: {eda_summary['filename']}
    Note: All monetary values are in Indian Rupees (₹). amounts are in absolute rupees, not lakhs or crores unless specified.
Dataset size: {shape[0]} rows × {shape[1]} columns
    

=== DOMAIN ANALYSIS ===
{domain_str}

=== KEY STATISTICS ===
{stats_str}

=== TOP CORRELATIONS ===
{corr_str}

Based on this data, provide your analysis in exactly this format:

## EXECUTIVE SUMMARY
(2-3 sentences covering what this dataset is and the most important finding)

## KEY INSIGHTS
(5 specific insights with exact numbers from the data)

## ANOMALIES & RED FLAGS
(3 things that are unusual, surprising, or worth investigating)

## RECOMMENDED NEXT STEPS
(3 concrete things an analyst should do next with this data)"""

    return prompt


prompt = build_prompt(eda_summary)
print('✅ Prompt built.')
print(f'   Prompt length: {len(prompt)} characters')
print(f'\n── PROMPT PREVIEW (first 500 chars) ──')
print(prompt[:500])

✅ Prompt built.
   Prompt length: 1730 characters

── PROMPT PREVIEW (first 500 chars) ──
I have analyzed a FUNDAMENTAL financial dataset: Annual_P_L_1_final.csv
    Note: All monetary values are in Indian Rupees (₹). amounts are in absolute rupees, not lakhs or crores unless specified.
Dataset size: 4668 rows × 57 columns


=== DOMAIN ANALYSIS ===
{
  "n_companies": 4668,
  "n_industries": 107,
  "profitable_pct": 77.9,
  "loss_making": 1008,
  "top_industries": {
    "Finance & Investments": 560,
    "Trading": 532,
    "Miscellaneous": 410,
    "Construction": 323,
    "Computers 


In [33]:
# ── Cell 5: Generate Insights ─────────────────────────────────────────────────
# Sends the prompt to Llama 3 via Ollama.
# Uses streaming so you see the response as it generates.

print('=' * 60)
print(f'  AUTOANALYST — LLM INSIGHTS')
print(f'  Dataset : {DATASET_FILENAME}')
print(f'  Model   : {MODEL}')
print('=' * 60)
print('Generating insights...\n')

stream = ollama.chat(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt}
    ],
    stream=True,
    options={
        'temperature': 0.3,   # low temp = more factual, less creative
        'num_predict': 1024,  # max tokens in response
    }
)

full_response = ''
for chunk in stream:
    token = chunk['message']['content']
    print(token, end='', flush=True)
    full_response += token

print('\n\n' + '=' * 60)
print('✅ Insights generated.')

  AUTOANALYST — LLM INSIGHTS
  Dataset : Annual_P_L_1_final.csv
  Model   : llama3:latest
Generating insights...

## EXECUTIVE SUMMARY
This dataset contains fundamental financial information for 4668 companies across 107 industries, with a total market capitalization of ₹44,464,893. The most striking finding is that 77.9% of companies are profitable, indicating a strong overall performance.

## KEY INSIGHTS
1. The mean current price of the companies is ₹602.74, with a standard deviation of ₹2985.70, indicating a relatively high level of price volatility. This could be an area of concern for investors.
2. The mean sales of the companies is ₹3676.53, with a standard deviation of ₹30,030.07, suggesting that sales are relatively high, but there is significant variation across companies.
3. The mean operating margin (OPM) is -8.95%, indicating that many companies are operating at a loss. This could be a sign of a challenging industry or a need for cost-cutting measures.
4. The mean profit a

In [25]:
# ── Cell 6: Save Insights Report ──────────────────────────────────────────────
# Saves the LLM output as a markdown report alongside the charts.

REPORTS_DIR = r'R:\AutoAnalyst\finance_module\outputs\reports'
os.makedirs(REPORTS_DIR, exist_ok=True)

report_name = DATASET_FILENAME.replace('.csv', '').replace(' ', '_').lower()
report_path = os.path.join(REPORTS_DIR, f'{report_name}_insights.md')

report_content = f"""# AutoAnalyst Insights Report
**Dataset:** {DATASET_FILENAME}
**Type:** {dataset_type.upper()}
**Model:** {MODEL}

---

{full_response}

---
*Generated by AutoAnalyst | Finance Module*
"""

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f'✅ Report saved → {report_path}')

✅ Report saved → R:\AutoAnalyst\finance_module\outputs\reports\credit_card_transactions_-_india_-_simple_insights.md


In [26]:
# ── Cell 7: Interactive Chat ───────────────────────────────────────────────────
# Ask follow-up questions about your data in natural language.
# The chat maintains context — it knows the dataset and the insights already generated.
# Type 'exit' or 'quit' to stop.

print('=' * 60)
print('  AUTOANALYST CHAT')
print(f'  Dataset: {DATASET_FILENAME}')
print('  Ask anything about your data. Type exit to quit.')
print('=' * 60)

# Build conversation history
# Start with the context the model already has
chat_history = [
    {'role': 'system',    'content': SYSTEM_PROMPT},
    {'role': 'user',      'content': prompt},
    {'role': 'assistant', 'content': full_response},
]

while True:
    user_input = input('\nYou: ').strip()

    if user_input.lower() in ['exit', 'quit', 'q', 'stop']:
        print('\n✅ Chat ended.')
        break

    if not user_input:
        continue

    # Add user message to history
    chat_history.append({'role': 'user', 'content': user_input})

    print('\nAnalyst: ', end='', flush=True)

    # Stream response
    stream = ollama.chat(
        model=MODEL,
        messages=chat_history,
        stream=True,
        options={'temperature': 0.3, 'num_predict': 512}
    )

    assistant_reply = ''
    for chunk in stream:
        token = chunk['message']['content']
        print(token, end='', flush=True)
        assistant_reply += token

    # Add assistant reply to history for next turn
    chat_history.append({'role': 'assistant', 'content': assistant_reply})
    print()

  AUTOANALYST CHAT
  Dataset: Credit card transactions - India - Simple.csv
  Ask anything about your data. Type exit to quit.

✅ Chat ended.
